# Task 3: Publish CPG Events sang Apache Kafka

Tài liệu này ghi nhận quá trình cấu hình, khởi tạo các topic và kiểm nghiệm tích hợp hệ thống publish event streaming của CPG Parser Service lên Apache Kafka broker chạy local.

## Kiến Trúc Kafka Event Streaming
- **Broker**: Apache Kafka chạy single-node ở chế độ KRaft (không ZooKeeper) trên Docker.
- **Topics**:
  - `cpg.nodes`: Chứa `NODE_UPSERT` và `NODE_DELETE`.
  - `cpg.edges`: Chứa `EDGE_UPSERT` và `EDGE_DELETE`.
  - `source.metadata`: Chứa `FILE_METADATA_UPSERT`.
  - `parser.errors`: Chứa `PARSER_ERROR` (Dead Letter Queue cho lỗi cú pháp).
- **Partition Key**: Sử dụng `file_id` làm khóa phân vùng để đảm bảo tính thứ tự tuần tự của các event thuộc cùng một file.
- **Guarantees**:
  - `acks=all` và producer có cấu hình `enable.idempotence=True`.
  - SQLite state database chỉ được commit sau khi nhận được delivery acknowledgement từ Kafka.

## 1. Thiết Lập Môi Trường & Import Thư Viện

In [1]:
import os
import subprocess
import json
import shutil
from pathlib import Path

# Resolve PROJECT_ROOT using git
PROJECT_ROOT = Path(
    subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
)

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /home/phat/AI_Project/lab04-cpg-streaming


## 2. Kiểm Trạng Thái Kafka Broker

In [2]:
# Verify that Kafka container is running and healthy
res_ps = subprocess.run(
    ["docker", "compose", "-f", str(PROJECT_ROOT / "infra/docker-compose.yml"), "ps"],
    check=True,
    capture_output=True,
    text=True
)
print(res_ps.stdout)


NAME        IMAGE                         COMMAND                  SERVICE   CREATED          STATUS                   PORTS
cpg-kafka   confluentinc/cp-kafka:7.4.0   "/etc/confluent/dock…"   kafka     10 minutes ago   Up 9 minutes (healthy)   0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp



## 3. Khởi Tạo Topics Idempotent

In [3]:
# Run topic creation script
res_topics = subprocess.run(
    [str(PROJECT_ROOT / "scripts/create_topics.sh")],
    check=True,
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT)
)
print(res_topics.stdout)


Waiting for Kafka broker to be healthy...
Kafka broker is healthy.
Creating topics idempotently...
Topic 'cpg.nodes' already exists.
Topic 'cpg.edges' already exists.
Topic 'source.metadata' already exists.
Topic 'parser.errors' already exists.
Topic 'connector.errors' already exists.

=== Existing Topics ===
__consumer_offsets
connector.errors
cpg.edges
cpg.nodes
parser.errors
source.metadata

=== Topic Details ===
Topic: cpg.nodes	TopicId: k4gnsSXGS22oxozK2a-O4Q	PartitionCount: 3	ReplicationFactor: 1	Configs: 
	Topic: cpg.nodes	Partition: 0	Leader: 1	Replicas: 1	Isr: 1
	Topic: cpg.nodes	Partition: 1	Leader: 1	Replicas: 1	Isr: 1
	Topic: cpg.nodes	Partition: 2	Leader: 1	Replicas: 1	Isr: 1
Topic: cpg.edges	TopicId: OD7XEFY-TtO-BEDZ-HSecw	PartitionCount: 3	ReplicationFactor: 1	Configs: 
	Topic: cpg.edges	Partition: 0	Leader: 1	Replicas: 1	Isr: 1
	Topic: cpg.edges	Partition: 1	Leader: 1	Replicas: 1	Isr: 1
	Topic: cpg.edges	Partition: 2	Leader: 1	Replicas: 1	Isr: 1
Topic: parser.errors	Top

## 4. Reset Smoke State cho Kafka Run
Chúng ta sẽ sử dụng một SQLite state database biệt lập để theo dõi trạng thái chạy.

In [4]:
KAFKA_SMOKE_STATE = PROJECT_ROOT / "workspace/state/notebook/kafka_publish_smoke.sqlite3"
if KAFKA_SMOKE_STATE.exists():
    KAFKA_SMOKE_STATE.unlink()

print("Isolated smoke state reset.")


Isolated smoke state reset.


## 5. Publish Smoke Events Lên Kafka
Chạy Parser Service ở live mode (`--no-dry-run`) giới hạn 1 file từ repository mục tiêu.

In [5]:
env = {**os.environ, "PARSER_STATE_DB": str(KAFKA_SMOKE_STATE)}

res_smoke = subprocess.run(
    [
        "uv", "run", "lab04", "parse-repository",
        "--scope", "smoke",
        "--limit", "1",
        "--no-dry-run"
    ],
    check=True,
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
    env=env
)
print(res_smoke.stdout)


Parsing repository (scope=smoke, limit=1, dry_run=False)...
Repository run completed. Summary:
{'discovered': 2779, 'eligible': 1, 'processed': 1, 'skipped_unchanged': 0, 'failed': 0, 'node_events': 1971, 'edge_events': 2417, 'metadata_events': 1, 'error_events': 0, 'duration_ms': 45268}



## 6. Inspect & Validate Messages từ Kafka
Tiến hành consume các message vừa được đẩy lên, kiểm tra partition key (`file_id`) và validate cấu trúc bằng JSON Schema.

In [6]:
res_inspect = subprocess.run(
    ["uv", "run", "python", "scripts/inspect_kafka_events.py"],
    check=True,
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
    env={**os.environ, "KAFKA_BOOTSTRAP_SERVERS": "localhost:9092"}
)
print(res_inspect.stdout)


Subscribing to topics: ['cpg.nodes', 'cpg.edges', 'source.metadata', 'parser.errors']
Listening for messages... (will auto-stop after 5s of inactivity)
[source.metadata] Part:0 Off:0 Key:526ba644702927c3bd46145a2903e7622db08269cc30575f09aa80925f5619bc Event:FILE_METADATA_UPSERT
  [OK] Schema validation passed.
[source.metadata] Part:0 Off:1 Key:526ba644702927c3bd46145a2903e7622db08269cc30575f09aa80925f5619bc Event:FILE_METADATA_UPSERT
  [OK] Schema validation passed.
[source.metadata] Part:0 Off:2 Key:526ba644702927c3bd46145a2903e7622db08269cc30575f09aa80925f5619bc Event:FILE_METADATA_UPSERT
  [OK] Schema validation passed.
[cpg.edges] Part:2 Off:0 Key:526ba644702927c3bd46145a2903e7622db08269cc30575f09aa80925f5619bc Event:EDGE_UPSERT
  [OK] Schema validation passed.
[cpg.edges] Part:2 Off:1 Key:526ba644702927c3bd46145a2903e7622db08269cc30575f09aa80925f5619bc Event:EDGE_UPSERT
  [OK] Schema validation passed.
[cpg.edges] Part:2 Off:2 Key:526ba644702927c3bd46145a2903e7622db08269cc30575f0

## 7. Kiểm Nghiệm Flow Lỗi Cú Pháp (Dead Letter Queue)
Chúng ta sẽ phân tích một tệp tin Python chứa lỗi cú pháp để kiểm tra xem `PARSER_ERROR` event có được đẩy vào topic `parser.errors` hay không, và đảm bảo state database của file này **không được commit**.

## 8. Consume & Validate Parser Error Event
Consume từ `parser.errors` topic để lấy `PARSER_ERROR` event và validate cấu trúc.

In [8]:
# Run verification tool (it will automatically poll all topics including parser.errors)
res_inspect_err = subprocess.run(
    ["uv", "run", "python", "scripts/inspect_kafka_events.py"],
    check=True,
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
    env={**os.environ, "KAFKA_BOOTSTRAP_SERVERS": "localhost:9092"}
)
print(res_inspect_err.stdout)


Subscribing to topics: ['cpg.nodes', 'cpg.edges', 'source.metadata', 'parser.errors']
Listening for messages... (will auto-stop after 5s of inactivity)
[source.metadata] Part:0 Off:0 Key:526ba644702927c3bd46145a2903e7622db08269cc30575f09aa80925f5619bc Event:FILE_METADATA_UPSERT
  [OK] Schema validation passed.
[source.metadata] Part:0 Off:1 Key:526ba644702927c3bd46145a2903e7622db08269cc30575f09aa80925f5619bc Event:FILE_METADATA_UPSERT
  [OK] Schema validation passed.
[source.metadata] Part:0 Off:2 Key:526ba644702927c3bd46145a2903e7622db08269cc30575f09aa80925f5619bc Event:FILE_METADATA_UPSERT
  [OK] Schema validation passed.
[cpg.edges] Part:2 Off:0 Key:526ba644702927c3bd46145a2903e7622db08269cc30575f09aa80925f5619bc Event:EDGE_UPSERT
  [OK] Schema validation passed.
[cpg.edges] Part:2 Off:1 Key:526ba644702927c3bd46145a2903e7622db08269cc30575f09aa80925f5619bc Event:EDGE_UPSERT
  [OK] Schema validation passed.
[cpg.edges] Part:2 Off:2 Key:526ba644702927c3bd46145a2903e7622db08269cc30575f0

## 9. Xác Minh Transaction Boundary
Xác minh xem state database của broken_syntax.py đã bị commit hay chưa (kết quả mong đợi là không tồn tại).

In [9]:
# Load state from state store using sqlite
import sqlite3
conn = sqlite3.connect(KAFKA_SMOKE_STATE)
cursor = conn.cursor()
cursor.execute("SELECT * FROM file_state WHERE file_path LIKE '%broken_syntax.py%'")
rows = cursor.fetchall()
conn.close()

print("Database committed rows for broken_syntax.py:", len(rows))
assert len(rows) == 0, "Error: State must not be committed for syntax error file"
print("SUCCESS: Transaction boundary verified.")


Database committed rows for broken_syntax.py: 0
SUCCESS: Transaction boundary verified.
